## tl;dr

The fixed `regime_trend` portfolio received **PROMOTE_RESEARCH** for
research use. On the locked 365-bar
holdout it returned **11.02%**, with Sharpe
**0.74** and maximum drawdown
**9.09%**. Forward status is
**WAITING_FOR_DATA**; this is not live-trading
approval.


## Context & Methods

This notebook independently reloads the fixed certificate, reruns the gate from
the checked-in Binance daily snapshots, and checks that the headline metrics and
all hard gates reconcile.

### Key Assumptions

- Five USDT markets use inverse-volatility budgets estimated from the prior
  60 completed daily returns, rebalanced every seven bars.
- Portfolio volatility is targeted at 20% annualized with a 100% gross cap;
  allocation changes incur 0.05% turnover cost.
- The last 365 completed UTC daily bars are the locked holdout.
- Commission, fixed/dynamic slippage, adverse funding scenarios, and parameter
  neighbours are included.
- Promotion means research acceptance only; live execution remains prohibited.
- The v1.1 contract is fingerprinted and needs 90 new completed daily bars after
  its frozen cutoff before it can request human review.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from backtest.investment_gate import (
    CERTIFICATE_PATH,
    evaluate_investment_gate,
    validate_gate_data,
)
saved = json.loads(CERTIFICATE_PATH.read_text(encoding="utf-8"))


## Data

In [2]:
quality = validate_gate_data()
print("data quality:", "PASS" if quality["passed"] else "FAIL")
for row in quality["datasets"]:
    print(
        row["symbol"],
        row["rows"],
        row["first_date"],
        row["last_date"],
        row["sha256"][:12],
    )


data quality: PASS
BTCUSDT 1000 2023-10-27 2026-07-22 504d54e4bf93
ETHUSDT 1000 2023-10-27 2026-07-22 56722d8c7a67
BNBUSDT 1000 2023-10-27 2026-07-22 b4a08adc8aa2
SOLUSDT 1000 2023-10-27 2026-07-22 fc1df4ee3741
XRPUSDT 1000 2023-10-27 2026-07-22 ef8fdcf45869


## Results

In [3]:
fresh = evaluate_investment_gate()
headline_fields = (
    "total_return_pct",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown_pct",
    "annualized_volatility_pct",
    "average_gross_exposure",
    "max_gross_exposure",
    "allocation_turnover",
    "median_asset_return_pct",
    "profitable_assets",
)
saved_headline = saved["evaluation"]["portfolio"]
fresh_headline = fresh["evaluation"]["portfolio"]
for field in headline_fields:
    assert saved_headline[field] == fresh_headline[field], field
print("decision:", fresh["decision"])
print({field: fresh_headline[field] for field in headline_fields})


decision: PROMOTE_RESEARCH
{'total_return_pct': 11.02, 'sharpe_ratio': 0.74, 'sortino_ratio': 1.07, 'max_drawdown_pct': 9.09, 'annualized_volatility_pct': 15.91, 'average_gross_exposure': 0.415, 'max_gross_exposure': 0.6, 'allocation_turnover': 1.799, 'median_asset_return_pct': 11.97, 'profitable_assets': 4}


In [4]:
failed = [gate for gate in fresh["gates"] if not gate["passed"]]
for gate in fresh["gates"]:
    print(
        "PASS" if gate["passed"] else "FAIL",
        gate["gate"],
        "value=", gate["value"],
        "threshold=", gate["threshold"],
    )
assert not failed, failed
assert fresh["live_trading_authorized"] is False


PASS data_quality value= True threshold= all structural checks pass
PASS portfolio_return value= 11.02 threshold= >= 5%
PASS portfolio_sharpe value= 0.74 threshold= >= 0.60
PASS portfolio_drawdown value= 9.09 threshold= <= 15%
PASS realized_volatility value= 15.91 threshold= <= 20% annualized
PASS development_stability value= {'return_pct': 9.37, 'sharpe_ratio': 0.47, 'max_drawdown_pct': 21.8} threshold= return > 0, Sharpe >= 0.30, drawdown <= 30%
PASS positive_asset_breadth value= 4 threshold= >= 3 of 5
PASS minimum_trades value= 54 threshold= >= 30
PASS benchmark_excess value= 31.07 threshold= >= 10 percentage points
PASS funding_stress value= {'funding_rate_pct': -0.01, 'return_pct': 9.7, 'max_drawdown_pct': 10.3} threshold= worst scenario return >= 5% and drawdown <= 25%
PASS allocation_gross_limit value= 0.6 threshold= <= 1.0
PASS parameter_sensitivity value= 9.14 threshold= all predeclared neighbours profitable


## Takeaways

- The strategy passes every declared research gate on the checked-in snapshot.
- The fully lagged risk budget cuts concentration and keeps realized volatility
  below its target; the notebook also reconciles development stability, funding
  stress, and parameter-neighbour gates.
- The evidence is still one 365-day holdout from one venue. Paper trading,
  capacity modelling, liquidation logic, and an independent future sample are
  required before any live-capital decision.
- Forward validation can only request human review; it cannot authorize live
  trading automatically.
